In [30]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import streamlit as st
import plotly.express as px

In [14]:
arquivo = r"C:\Users\gabri\Desktop\FIAP\Fase_5\Tech_Challenge\data\BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

In [15]:
def limpar_colunas(df):
    # remove espaços invisíveis
    df.columns = df.columns.str.strip()

    # remove duplicadas mantendo a primeira
    df = df.loc[:, ~df.columns.duplicated()]

    return df

In [16]:
def padronizar(df, ano):

    df = limpar_colunas(df)

    # mantém apenas o INDE do ano atual
    col_inde = [c for c in df.columns if "INDE" in c and str(ano)[-2:] in c or str(ano) in c]

    if len(col_inde) > 0:
        df["INDE"] = df[col_inde[0]]
    else:
        df["INDE"] = None

    mapa = {
        "Nome": "NOME",
        "Nome Anonimizado": "NOME",
        "IAN": "IAN",
        "IDA": "IDA",
        "IEG": "IEG",
        "IAA": "IAA",
        "IPS": "IPS",
        "IPP": "IPP",
        "IPV": "IPV",
    }

    df = df.rename(columns=mapa)

    COLUNAS_PADRAO = [
        "RA", "NOME", "IAN", "IDA", "IEG",
        "IAA", "IPS", "IPP", "IPV", "INDE"
    ]

    for col in COLUNAS_PADRAO:
        if col not in df.columns:
            df[col] = None

    df = df[COLUNAS_PADRAO].copy()
    df["ANO"] = ano

    return df


In [17]:
df_2022 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2022"), 2022)
df_2023 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2023"), 2023)
df_2024 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2024"), 2024)

In [18]:
for nome, df_temp in {
    "2022": df_2022,
    "2023": df_2023,
    "2024": df_2024
}.items():

    duplicadas = df_temp.columns[df_temp.columns.duplicated()]
    print(nome, "-> duplicadas:", list(duplicadas))


2022 -> duplicadas: []
2023 -> duplicadas: []
2024 -> duplicadas: []


In [19]:
df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

In [20]:
df.head()

,RA,NOME,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO
0,RA-1,Aluno-1,5.0,4.0,4.1,8.3,5.6,None,7.278,5.783,2022
1,RA-2,Aluno-2,10.0,6.8,5.2,8.8,6.3,None,6.778,7.055,2022
2,RA-3,Aluno-3,10.0,5.6,7.9,0.0,5.6,None,7.556,6.591,2022
3,RA-4,Aluno-4,10.0,5.0,4.5,8.8,5.6,None,5.278,5.951,2022
4,RA-5,Aluno-5,10.0,5.2,8.6,7.9,5.6,None,7.389,7.427,2022


In [ ]:
df = df.sort_values(["RA", "ANO"])

df["delta_IDA"] = df.groupby("RA")["IDA"].diff()
df["delta_INDE"] = df.groupby("RA")["INDE"].diff()

In [24]:
df.head(10)

,RA,NOME,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE
0,RA-1,Aluno-1,5.0,4.00,4.100000,8.300,5.60,None,7.278,5.783,2022,NaN,NaN
1855,RA-1,Aluno-1,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN
2958,RA-1,Aluno-1,10.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,2024,NaN,NaN
9,RA-10,Aluno-10,5.0,4.10,5.200000,8.300,5.00,None,7.056,5.784,2022,NaN,NaN
99,RA-100,Aluno-100,10.0,7.60,7.800000,8.800,5.00,None,7.250,7.618,2022,NaN,NaN
1072,RA-1000,Aluno-1000,10.0,7.00,9.400000,8.500,3.77,6.25,8.920,7.9162,2023,NaN,NaN
2225,RA-1000,Aluno-1000,10.0,7.75,9.545455,9.002,6.26,8.125,7.835,8.364791,2024,0.75,0.448591
1074,RA-1001,Aluno-1001,5.0,7.80,9.100000,9.000,7.52,7.5,9.170,8.1162,2023,NaN,NaN
2227,RA-1001,Aluno-1001,5.0,7.75,9.347826,7.502,7.51,7.916667,7.920,7.796432,2024,-0.05,-0.319768
1075,RA-1002,Aluno-1002,5.0,7.00,9.700000,9.000,7.52,6.25,8.920,7.9012,2023,NaN,NaN


In [25]:
df["risco"] = (df["delta_INDE"] < -1).astype(int)

In [28]:


features = ["IAN", "IDA", "IEG", "IAA", "IPS", "IPP", "IPV"]

X = df[features].fillna(0)
y = df["risco"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Acurácia:", model.score(X_test, y_test))


Acurácia: 0.9504950495049505


In [ ]:
df["prob_risco"] = model.predict_proba(X)[:, 1]

In [35]:

df.columns

Index(['RA', 'NOME', 'IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'INDE',
       'ANO', 'delta_IDA', 'delta_INDE', 'risco', 'prob_risco'],
      dtype='object')

In [38]:
df.to_csv(r"C:\Users\gabri\Desktop\FIAP\Fase_5\Tech_Challenge\data\base_final.csv")